In [ ]:
# Viewer: get the code, compile, start the drawing program and its web server, show the page.
# Run this cell again after every push. Needs a T4 GPU runtime.
import os, shutil, signal, subprocess, sys, time, urllib.request

REPO_URL = "https://github.com/ShreyasxAnand/pointwalk"
BRANCH = "dev"
FLAGS = ["-O2", "-arch=sm_75", "-fmad=false"]  # -fmad=false: CPU and GPU match bit for bit
PORT = 8000

SCENE = "waymo"   # "waymo": one real lidar frame.  "synthetic": the floor and ball from project.cu.

# Only used when SCENE is "waymo". Same settings as in the launcher notebook.
WAYMO_INPUT = "gs://waymo_open_dataset_v_2_0_1/training"
WAYMO_SEGMENT = "10023947602400723454_1120_000_1140_000"
WAYMO_FRAME = 0                        # which frame to look at
WAYMO_FRAMES = 2                       # how many frames to convert; must be more than WAYMO_FRAME
WAYMO_OUT = "/content/waymo_out/" + WAYMO_SEGMENT   # the converted files, kept outside the repo
WAYMO_PKGS = "/content/waymo_pkgs"     # the converter's pinned packages, kept apart from Colab's own

repo = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
build = os.path.join(repo, "build")
pid_file = "/content/pointwalk_viewer.pid"
log_file = "/content/pointwalk_viewer.log"


def sh(*cmd, cwd=None, quiet=False, env=None):
    """Run a command, print its output, stop the cell if it fails."""
    print("$ " + " ".join(cmd), flush=True)
    r = subprocess.run(cmd, cwd=cwd, env=env, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if not quiet or r.returncode != 0:
        print(r.stdout, flush=True)
    if r.returncode != 0:
        raise SystemExit(f"FAILED (exit {r.returncode}): {' '.join(cmd)}")
    return r.stdout


# 0. Stop the server left over from the last time this cell ran (it holds the port and the GPU).
if os.path.exists(pid_file):
    try:
        os.killpg(int(open(pid_file).read()), signal.SIGTERM)
        time.sleep(1)
    except (ProcessLookupError, ValueError, PermissionError):
        pass
    os.remove(pid_file)

# 1. Get the code.
if not os.path.isdir(os.path.join(repo, ".git")):
    sh("git", "clone", "--branch", BRANCH, REPO_URL, repo)
else:
    sh("git", "fetch", "origin", BRANCH, cwd=repo)
    sh("git", "checkout", BRANCH, cwd=repo)
    sh("git", "pull", "--ff-only", "origin", BRANCH, cwd=repo)
sh("git", "log", "-1", "--format=%h %ad %s", "--date=iso", cwd=repo)  # what is being run
sh("nvidia-smi", "--query-gpu=name", "--format=csv,noheader")

# 2. Compile into a fresh build/, and make the scene file the viewer will show.
shutil.rmtree(build, ignore_errors=True)
os.makedirs(build)
maker = "range_to_points" if SCENE == "waymo" else "project"   # the program that saves the scene file
for name in [maker, "viewer"]:
    sh("nvcc", *FLAGS, os.path.join(repo, "src", name + ".cu"), "-o", os.path.join(build, name))

if SCENE == "waymo":
    # Convert a few frames straight from Waymo's bucket, only if they are not there yet.
    assert WAYMO_FRAME < WAYMO_FRAMES, "WAYMO_FRAMES must be more than WAYMO_FRAME"
    last_file = os.path.join(WAYMO_OUT, "range", f"frame{WAYMO_FRAMES - 1:04d}_laser1.bin")
    if not os.path.exists(last_file):
        from google.colab import auth
        auth.authenticate_user()  # sign in with the Google account that has access to the Waymo dataset
        sh(sys.executable, "-m", "pip", "install", "-q", "--target", WAYMO_PKGS,
           "-r", os.path.join(repo, "waymo", "requirements.txt"), quiet=True)
        sh(sys.executable, os.path.join(repo, "waymo", "convert.py"), "--input", WAYMO_INPUT,
           "--segment", WAYMO_SEGMENT, "--out", WAYMO_OUT, "--frames", str(WAYMO_FRAMES),
           env={**os.environ, "PYTHONPATH": WAYMO_PKGS})
    # One range image -> dots (your step 7 kernel). It saves them as a scene file.
    out = sh("./range_to_points", WAYMO_OUT, str(WAYMO_FRAME), cwd=build, quiet=True)
    print("\n".join(line for line in out.splitlines() if "return" in line or "differ" in line or "saved" in line))
    scene_file = os.path.join(build, f"frame{WAYMO_FRAME:04d}_laser1_points.bin")
else:
    out = sh("./project", cwd=build, quiet=True)
    print("\n".join(line for line in out.splitlines() if "scene" in line or "differ" in line))
    scene_file = os.path.join(build, "scene.bin")

# 3. Start the web server. It starts the drawing program itself and keeps running after this cell ends.
log = open(log_file, "w")
server = subprocess.Popen(
    [sys.executable, os.path.join(repo, "viewer", "server.py"),
     "--renderer", os.path.join(build, "viewer"),
     "--scene", scene_file,
     "--port", str(PORT), "--host", "0.0.0.0"],   # reachable only through Colab's own signed-in link
    stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
open(pid_file, "w").write(str(server.pid))

ready = False
for _ in range(60):                      # wait up to 30 s for it to answer
    if server.poll() is not None:
        break
    try:
        ready = urllib.request.urlopen(f"http://127.0.0.1:{PORT}/ping", timeout=1).read() == b"ok"
        if ready:
            break
    except OSError:
        time.sleep(0.5)
print(open(log_file).read())
if not ready:
    raise SystemExit("The viewer server did not start. The lines above are its log.")

# 4. One picture straight from the server, to prove the GPU side works before the page is involved.
reply = urllib.request.urlopen(f"http://127.0.0.1:{PORT}/frame?reset=1", timeout=10)
print("first picture:", len(reply.read()), "bytes,",
      "kernels", reply.headers.get("X-Kernel-Ms"), "ms,",
      "whole picture on GPU", reply.headers.get("X-Frame-Ms"), "ms")

# 5. Show the page. Click the picture first, then use the keys.
if SCENE == "waymo":
    print("You start on the road under the car, facing forward. Hold E to rise: the laser is about 2 m up.")
from google.colab import output
try:
    output.serve_kernel_port_as_window(PORT, anchor_text="Open the viewer in its own tab (use this if the box below stays empty)")
except Exception as problem:
    print("could not make the link:", problem)
try:
    output.serve_kernel_port_as_iframe(PORT, height="640")
except Exception as problem:
    print("could not show the page here:", problem)
